# 08 · Report: all version-2 tables

Collects the results of notebooks 01-07 (on Kaggle: add their outputs as inputs; locally
they are read from `results/version_2/`) and writes every table as CSV plus one
`REPORT.md`. No training.

## Options

In [ ]:
NAME = "08_report"

NOTEBOOKS = ["02_tabular_benchmark", "03_multievent_benchmark", "04_ablation_encoding_representation",
             "05_ablation_losses", "06_regression_head", "07_llm_end_to_end"]

# --- execution ------------------------------------------------------------------------
SMOKE_TEST   = False    # True: 2 epochs, 1 seed, 1-2 configurations - only to check that it runs
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h);
                            # re-run with this notebook's output attached to continue
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # the code is cloned from here
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    RESULTS_ROOT = WORK / "results"
    INPUT_ROOTS = [Path("/kaggle/input")]
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "version_2"
    INPUT_ROOTS = []
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
print("repo:", REPO, "| commit:", COMMIT, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy and torch are NEVER moved: pandas/pyarrow and CUDA in the image are built for them.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            "pycox==0.3.0", "peft", "easydict", "lifelines", "osqp", "patsy",
            f"numpy=={_np.__version__}", f"torch=={torch.__version__.split('+')[0]}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    # SurvivalEVAL declares torch>=2.13; it runs on older torch, so its pins are skipped
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "SurvivalEVAL==0.8.5"], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
import pandas as pd
from scripts.runner import Runner, select_best
from scripts import v2

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input") and finished runs are
# copied back, so only the missing ones are trained.
def _incomplete(f):
    """A survival result scored before the IBS/IBLL/AUC fix (commit after b0bbe25): rerun it."""
    if f.name.endswith(".meta.json") or f.name == "environment.json" or not f.is_file():
        return False
    test = json.loads(f.read_text()).get("test", {})
    return "ctd_weighted_avg" in test and "ibs" not in test

n_copied, n_redo = 0, 0
for root in INPUT_ROOTS:
    for d in root.rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if (r.results_dir / f.name).exists():
                    continue
                if _incomplete(f) or _incomplete(d / f.name.replace(".meta.json", ".json")):
                    n_redo += f.name.endswith(".meta.json") is False
                    continue
                shutil.copy2(f, r.results_dir / f.name); n_copied += 1
print(f"resumed {n_copied} files from attached inputs; {n_redo} runs without IBS will be rerun")
(r.results_dir / "environment.json").write_text(json.dumps({
    "commit": COMMIT, "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Collect

In [ ]:
from scripts.runner import Runner
frames = {}
for nb in NOTEBOOKS:
    dirs = [RESULTS_ROOT / nb] + [d for root in INPUT_ROOTS for d in root.rglob(nb) if d.is_dir()]
    for d in dirs:
        if any(d.glob("*.meta.json")):
            df = Runner(repo=REPO, results_dir=d, verbose=False).collect()
            df["notebook"] = nb
            frames[nb] = df
            break
    print(f"{nb:38s} {len(frames.get(nb, [])):5d} runs")
allres = pd.concat(frames.values(), ignore_index=True) if frames else pd.DataFrame()
out = r.results_dir
allres.to_csv(out / "all_runs.csv", index=False)

## Tables

In [ ]:
lines = ["# Version 2 results", "", f"code commit {COMMIT}", ""]
def add(title, df, fname):
    if df is None or not len(df):
        return
    df.to_csv(out / fname, index=False)
    lines.extend([f"## {title}", "", df.to_markdown(index=False), ""])
    display(Markdown(f"### {title}")); display(df)

if "02_tabular_benchmark" in frames:
    f = frames["02_tabular_benchmark"]
    add("§4.1 C_td at 25/50/75% (SurvTRACE Table 2 layout)", v2.horizon_table(f, ["dataset", "model"]), "t_tabular_horizons.csv")
    add("§4.1 all metrics", v2.table(f, ["dataset", "model"]), "t_tabular_all.csv")
if "03_multievent_benchmark" in frames:
    f = frames["03_multievent_benchmark"]
    for name in sorted(f.dataset.unique()):
        K = v2.DATASETS[name]["events"]
        add(f"§4.3 {name}: C_td per event", v2.horizon_table(f[f.dataset == name], ["model"], K=K), f"t_multi_{name}.csv")
    add("§4.3 all metrics", v2.table(f, ["dataset", "model"]), "t_multi_all.csv")
if "04_ablation_encoding_representation" in frames:
    add("§2.2 encoding and representation", v2.table(frames["04_ablation_encoding_representation"], ["dataset", "variant"]), "t_encoding_representation.csv")
if "05_ablation_losses" in frames:
    add("§2.3 loss combinations", v2.table(frames["05_ablation_losses"], ["dataset", "variant"]), "t_losses.csv")
if "06_regression_head" in frames:
    add("§2.4 regression head", v2.table(frames["06_regression_head"], ["dataset", "variant"]), "t_regression.csv")
if "07_llm_end_to_end" in frames:
    add("§3 end-to-end LLM", v2.table(frames["07_llm_end_to_end"], ["dataset", "variant"]), "t_llm.csv")
(out / "REPORT.md").write_text("\n".join(lines), encoding="utf-8")
print("zip:", r.zip())

## Clean up

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)